# Day 36：里程碑项目 2 Part 1 — 企业知识库问答系统后端 (`EnterpriseKBService`)

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段 Day 36！今天我们将实战构建企业知识库问答系统后端（`v0.3.0-day36`），打通 `.txt`/`.md`/`.csv` 多格式文档解析切分、BM25 + 向量混合检索、RRF 融合与 Rerank 精排、带引用溯源的安全问答，以及 SQLite 多轮会话持久化。

## 1. 环境初始化与导入 `project_iterations/day_36` 模块
自动定位仓库根目录并将 `project_iterations/day_36` 加入 `sys.path`，导入 `EnterpriseKBService`。

In [1]:
import sys
import tempfile
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day36_dir = repo_root / "project_iterations" / "day_36"
if str(day36_dir) not in sys.path:
    sys.path.insert(0, str(day36_dir))

from zhishu_agent import EnterpriseKBService

print("已加载模块路径:", day36_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_36


## 2. 多格式文档入库（`.txt` / `.md` / `.csv`）与切片查看
在临时目录中构造三份典型企业文档（HR 制度、SRE 运维手册、差旅预算表），使用 `ingest_file` 解析入库。

In [2]:
tmp_obj = tempfile.TemporaryDirectory()
tmp_dir = Path(tmp_obj.name)

txt_path = tmp_dir / "hr_policy.txt"
txt_path.write_text(
    "星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。申请年假需提前3个工作日在OA系统审批。",
    encoding="utf-8",
)

md_path = tmp_dir / "sre_runbook.md"
md_path.write_text(
    "# Redis 运维规范\n生产环境 Redis 集群严禁执行 KEYS * 命令，请使用 SCAN 分批遍历。\n\n"
    "## 网关故障码 ERR_502\n遇到 ERR_502 网关错误时，请检查上游 Pod 健康状态并执行滚动重启。",
    encoding="utf-8",
)

csv_path = tmp_dir / "travel_budget.csv"
csv_path.write_text(
    "城市等级,住宿上限,餐饮补贴\n一线城市,600元/晚,150元/天\n二线城市,450元/晚,120元/天\n",
    encoding="utf-8",
)

db_path = tmp_dir / "kb_sessions.db"
kb = EnterpriseKBService(db_path=db_path, chunk_size=200, chunk_overlap=30)
for f in (txt_path, md_path, csv_path):
    res = kb.ingest_file(f)
    print(f"入库成功: {res.source} ({res.file_type}) -> {res.num_chunks} 个切片, doc_id={res.doc_id}")

入库成功: hr_policy.txt (.txt) -> 1 个切片, doc_id=doc-c6db83df17
入库成功: sre_runbook.md (.md) -> 2 个切片, doc_id=doc-5c4383a778
入库成功: travel_budget.csv (.csv) -> 2 个切片, doc_id=doc-4f76dbe28a


## 3. BM25 + 向量混合检索、RRF 融合与 Rerank 精排
查看 `retrieve()` 返回的 `bm25_score`、`vector_score`、`rrf_score` 与 `rerank_score`。

In [3]:
hits = kb.retrieve("遇到 ERR_502 网关错误怎么处理？", top_k=2, use_rerank=True)
for i, h in enumerate(hits, start=1):
    print(
        f"Top-{i} [{h.chunk.source}#{h.chunk.chunk_index}] "
        f"rerank={h.rerank_score:.4f} | rrf={h.rrf_score:.4f} | "
        f"bm25={h.bm25_score:.4f} | vec={h.vector_score:.4f}"
    )
    print("   内容:", h.chunk.content.replace("\n", " "))

Top-1 [sre_runbook.md#1] rerank=0.6223 | rrf=0.0328 | bm25=19.1661 | vec=0.3671
   内容: ## 网关故障码 ERR_502 遇到 ERR_502 网关错误时，请检查上游 Pod 健康状态并执行滚动重启。
Top-2 [hr_policy.txt#0] rerank=0.0185 | rrf=0.0161 | bm25=0.0000 | vec=0.0669
   内容: 星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。申请年假需提前3个工作日在OA系统审批。


## 4. 引用溯源回答、安全拒答与 SQLite 多轮会话持久化
执行库内提问与库外离题提问，然后关闭服务并重新连接同一 SQLite 数据库验证历史无损恢复。

In [4]:
sid = "nb-session-01"
resp1 = kb.ask("一线城市差旅住宿上限是多少？", session_id=sid, top_k=2)
print("=== 库内提问（带引用溯源）===")
print(f"grounded={resp1.grounded}")
print(resp1.format_with_sources())

resp2 = kb.ask("量子计算机的退相干时间是多少？", session_id=sid, top_k=2)
print("\n=== 库外提问（安全拒答）===")
print(f"grounded={resp2.grounded} | answer={resp2.answer}")
kb.close()

# 重新打开 SQLite 数据库，验证会话历史持久化
kb_reopened = EnterpriseKBService(db_path=db_path)
history = kb_reopened.get_session_history(sid)
print(f"\n=== SQLite 持久化恢复 ({len(history)} 条消息) ===")
for msg in history:
    print(f"[{msg.role}] {msg.content} (citations={len(msg.citations)})")
kb_reopened.close()
tmp_obj.cleanup()

=== 库内提问（带引用溯源）===
grounded=True
根据企业知识库检索结果：[travel_budget.csv 第1行] 城市等级: 一线城市 | 住宿上限: 600元/晚 | 餐饮补贴: 150元/天 [1] [travel_budget.csv 第2行] 城市等级: 二线城市 | 住宿上限: 450元/晚 | 餐饮补贴: 120元/天 [2]

参考来源：
[1] travel_budget.csv (chunk #0, score=0.5923): [travel_budget.csv 第1行] 城市等级: 一线城市 | 住宿上限: 600元/晚 | 餐饮补贴: 150元/天
[2] travel_budget.csv (chunk #1, score=0.5382): [travel_budget.csv 第2行] 城市等级: 二线城市 | 住宿上限: 450元/晚 | 餐饮补贴: 120元/天

=== 库外提问（安全拒答）===
grounded=False | answer=抱歉，当前企业知识库中未检索到与您问题直接相关的依据，无法给出确切回答。

=== SQLite 持久化恢复 (4 条消息) ===
[user] 一线城市差旅住宿上限是多少？ (citations=0)
[assistant] 根据企业知识库检索结果：[travel_budget.csv 第1行] 城市等级: 一线城市 | 住宿上限: 600元/晚 | 餐饮补贴: 150元/天 [1] [travel_budget.csv 第2行] 城市等级: 二线城市 | 住宿上限: 450元/晚 | 餐饮补贴: 120元/天 [2] (citations=2)
[user] 量子计算机的退相干时间是多少？ (citations=0)
[assistant] 抱歉，当前企业知识库中未检索到与您问题直接相关的依据，无法给出确切回答。 (citations=0)
